### sys.executable > 확인 역할 노트북이 어느 파이썬에 있는지 설치 위치를 확인함 사실 이미 가상환경이 하나 더 있어서 위치가 처음에 조금 헷갈렸었음

In [5]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

c:\Users\akals\Downloads\model-serving-course\.venv\Scripts\python.exe
2.12.0+cpu


## 모델정의
모델 구조를 간단히 정의, 구현. 
은닉층 하나짜리인 간단한 모델을 구현해서 파이프라인을 검증하기 위한 재료로 씀
파이프라인 검증이니 모델을 복잡하게 하면 변인 통제가 제대로 되지 않지 않을까 싶음 


In [8]:
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(x)

## 데이터 로드&학습


In [17]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_data = datasets.MNIST(
    root = "../data", train=True, download=True,
    transform=transforms.ToTensor()
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

model = SimpleClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(1):
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch}: loss {loss.item():.4f}")

          

epoch 0: loss 0.3514


In [13]:
import os
%pip install onnxscript
os.makedirs("../models", exist_ok=True)


   ---------------------------------------- 0.0/714.8 kB ? eta -:--:--
   --------------------------------------- 714.8/714.8 kB 13.1 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [onnx_ir]
   ---------------------------------------- 0/2 [onnx_ir]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   -------------------- ------------------- 1/2 [onnxscript]
   ---------------------------------------- 2/2 [onnxscript]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 3가지 방식으로 저장 

model.eval()는 모델의 모드 전환 스위치.  안쳐도 상관은 없었지만 습관으로 만들기 위해서 추가함 
example_input는 2,3이 공유하는 재료로 가짜 입력을 실제로 모델에 흘려보내면서 연산경로를 추적함. 

torch.save를 통해 각 경로로 저장되며 
torch.save(①), ts.save(②), torch.onnx.export(③) — 담는 내용이 다르니 저장 도구도 다르다. 
①은 가중치 딕셔너리만, 
②는 걸어간 연산 경로(구조)까지, 
③은 PyTorch 밖에서도 열리도록 표준 포맷으로 번역해 내보냄.

In [19]:
model.eval()

example_input = torch.randn(1, 1, 28, 28)

#가중치 저장
torch.save(model.state_dict(), "../models/model.pth")
#구조 저장 
ts = torch.jit.trace(model, example_input)
ts.save("../models/model.pt")
#오닉스 - 프레임워크 독립
torch.onnx.export(
    model, example_input, "../models/model.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

C:\Users\akals\AppData\Local\Temp\ipykernel_58856\1531014663.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


C:\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.12.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch_size,1,28,28]>
            ),
            outputs=(
                %"output"<FLOAT,[batch_size,10]>
            ),
            initializers=(
                %"fc.1.weight"<FLOAT,[128,784]>{TorchTensor(...)},
                %"fc.1.bias"<FLOAT,[128]>{TorchTensor(...)},
                %"fc.3.weight"<FLOAT,[10,128]>{TorchTensor(...)},
                %"fc.3.bias"<FLOAT,[10]>{TorchTensor<FLOAT,[10]>(Parameter containing: tensor([-0.1247,  0.0674,  0.0348, -0.0030, -0.0163,  0.1166, -0.0712, -0.0242, -0.1247,  0.0300], requires_grad=True), name='fc.3.bias')},
                %"val_3"<INT64,[1]>{Tensor<INT64,[1]>(array([784]), name='val_3')}
      

In [1]:
import os
print(os.listdir("../models"))

['model.onnx', 'model.onnx.data', 'model.pt', 'model.pth']


In [2]:
print(os.getcwd())
print(os.listdir("../models"))

c:\Users\akals\Downloads\model-serving-course\.venv
['model.onnx', 'model.onnx.data', 'model.pt', 'model.pth']


In [3]:
import os
for f in sorted(os.listdir("../models")):
    print(f"{f:20s} {os.path.getsize(os.path.join('../models', f)):>10,} bytes")

model.onnx                8,213 bytes
model.onnx.data         407,040 bytes
model.pt                418,405 bytes
model.pth               409,353 bytes


### 유사배포단계? 
import onnxruntime as ort는 사실 torch.onnx와는 별개의 패키지 같아 보였음 
torch.onnx가 파이토치에서 오닉스로 변환하는 출구역할이라면 onnxruntime 는 실행하는 쪽의 독립엔진이다. 
분리 : 배포지에는 onnxruntime깔면 됨, 서버에서 코드가 그대로 돌게 됨 
다만 torch.Tensor를 그대로 박을수는 없기 때문에 numpy로 변환해야 한다. 

검증은 np.allclose로 출력을 비교하고 
호출자가 나 > 남으로 바뀌는 순간이다. 


In [13]:
import onnxruntime as ort
import numpy as np

test_input = torch.randn(4, 1, 28, 28)

# 클래스 정의로 
m1 = SimpleClassifier()
m1.load_state_dict(torch.load("../models/model.pth", weights_only=True))
m1.eval()
with torch.no_grad():
    out1 = m1(test_input)

#파일로 로드

m2 = torch.jit.load("../models/model.pt")
m2.eval()
with torch.no_grad():
    out2 = m2(test_input)

# 오닉스

sess = ort.InferenceSession("../models/model.onnx")
out3 = sess.run(None, {"input": test_input.numpy()})[0]

print("pth vs pt  :", np.allclose(out1.numpy(), out2.numpy()))
print("pth vs onnx:", np.allclose(out1.numpy(), out3, atol=1e-5))

pth vs pt  : True
pth vs onnx: True


In [14]:
def predict(model, x):
    model.eval()
    with torch.no_grad():
        return model(x).argmax(dim=1)

print(predict(m1, test_input))   # 배치 4개의 예측 클래스

tensor([7, 0, 2, 6])


In [15]:
from torchvision import datasets, transforms
test_data = datasets.MNIST(root="../data", train=False, download=True,
                           transform=transforms.ToTensor())
x, y = next(iter(torch.utils.data.DataLoader(test_data, batch_size=8)))
print("예측:", predict(m1, x).tolist())
print("정답:", y.tolist())

예측: [7, 2, 1, 0, 4, 1, 4, 9]
정답: [7, 2, 1, 0, 4, 1, 4, 9]


### 2일차 - FastAPI 기초와 데이터 처리 
노트북 속 모델을 HTTP로 부를수 있는  API로 감싼다. 

In [13]:
# 셀1
import os 
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(os.getcwd())   # ...\model-serving-course 확인

c:\Users\akals\Downloads\model-serving-course


### 작업위치를 고정
어제한 작업이 계속 파일을 읽어오지 못해서 작업위치를 아예 고정하기로 함
주피터는 이미 이벤트 루프가 돌아서 uvicorn을 그냥 못 띄운다, 그래서 별도 스레드

In [14]:
# [셀 1] 작업 위치를 프로젝트 루트로 고정
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(os.getcwd())   # ...\model-serving-course 확인

# [셀 2] 어제 가중치가 오늘 구조와 맞는지 검증
import torch
sd = torch.load("models/model.pth", weights_only=True)
for k, v in sd.items():
    print(k, tuple(v.shape))

c:\Users\akals\Downloads\model-serving-course
fc.1.weight (128, 784)
fc.1.bias (128,)
fc.3.weight (10, 128)
fc.3.bias (10,)


### Fast API 서버의 최소단위 > FastAPI()객체 하나, 데코레이터 하나, 함수 하나, dict리턴, 기본 구조라 봐야할듯 
@app.get("/health") > 데코레이터 하나가 메서드+경로 두가지를 동시에 선언함 
dict 을 그냥 리턴하면 JSON변환은 FastAPI가 한다는 것 


### 오류1 

%%writefile를 IPython이 인식하지 못함 
첫줄 위치 교정을 시도해도 여전히 not found가 나왔고 
%lsmagic으로 확인했을 때 %%writefile이 존재하는건 확인함
새 셀에 다시 쳤을 때 에러가 바뀌었고 os.makedirs('app')으로 성공함 

%%명령은 반드시 셀의 첫 줄에 있어야함. 
근데 나는 주석으로 #셀4라고 가장 첫칸에 달아놨기 때문에 오류가 생겼던것 같음



In [21]:
#셀 4 

%%writefile app/main_basic.py

from fastapi import FastAPI

app = FastAPI(title="My First ML API")

@app.get("/health")
def health_check():
    return {"status": "healthy"}

UsageError: Line magic function `%%writefile` not found.


In [22]:
%lsmagic

Available line magics:
%alias  %alias_magic  %autoawait  %autocall  %automagic  %autosave  %bookmark  %cd  %clear  %cls  %code_wrap  %colors  %conda  %config  %connect_info  %copy  %ddir  %debug  %dhist  %dirs  %doctest_mode  %echo  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %ls  %lsmagic  %macro  %magic  %mamba  %matplotlib  %micromamba  %mkdir  %more  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %pip  %popd  %pprint  %precision  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %ren  %rep  %rerun  %reset  %reset_selective  %rmdir  %run  %save  %sc  %set_env  %store  %subshell  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %uv  %who  %who_ls  %whos  %xdel  %xmode

Available cell magics:
%%!  %%HTML  %%SVG  %%bash  %%capture  %%cmd  %%code_wrap  %%debug  %%file  %%html  %%javascript

In [24]:
%%writefile app/test_magic.py
print("hello")

Writing app/test_magic.py


FileNotFoundError: [Errno 2] No such file or directory: 'app/test_magic.py'

In [ ]:
import os
os.makedirs('app', exist_ok=True)   # 있으면 넘어가고, 없으면 만들고

In [27]:
%%writefile app/main_basic.py
# 최소 서버 작성
from fastapi import FastAPI

app = FastAPI(title="My First ML API")

@app.get("/health")
def health_check():
    return {"status": "healthy"}

Writing app/main_basic.py


### 서버 실행 도우미
사실 노트북에서만 구현하는 내용이라 굳이 손으로 쳐넣을 필요가 없었는데 굳이 손으로 일일히 쳐넣어서 고생을 했음.. 
그냥 써놓기만 하면 되긴 했는데 사실 import에서 몇번 오류가 생겨서 추가로  pip을 군데군데 넣어야 하기도 했고 
코드 오류코드를 잡기 위해서 일일히 오류코드를 타고 코드 번호를 찾아가면서 일일히 원인이 뭔지 찾아야 했다. 
굳이 배운게 있다면 그런 디버깅 과정이 아니었을까... 

사건 3: 도우미 셀 손타이핑 — 버그 7개

원본을 손으로 치면서 들어간 것들: hetcwd(오타), soket(오타), settimeout(0,5)(쉼표≠소수점, 인자 2개가 됨), sys.module(s 누락), log_level='Warning'(uvicorn은 소문자만), 대기 루프 들여쓰기가 if 블록 안으로, ProactorEventLoop(원본은 Selector)
특이사항: hetcwd는 and 단축평가(short-circuit) 덕에 현재 환경에서는 실행조차 안 돼 잠복 — 에러가 안 난다 ≠ 버그가 없다
진행: 에러 메시지 3단계 읽기(메시지 해부 → Traceback 위치 추적 → 의도 복원)로 sys.modules, settimeout(0.5) 직접 수정

사건 4: 에러 없이 조용히 죽는 서버

증상: Traceback 없이 "서버 실행 실패"만 출력. 별도 스레드 안에서 죽으면 비명이 셀 출력으로 올라오지 않는다.
진단법: 스레드를 우회해 서버가 import하는 대상을 노트북에서 직접 import → ModuleNotFoundError: No module named 'fastapi' 자백 확보
원인: 오늘 필요한 신규 패키지 둘(fastapi, uvicorn) 중 uvicorn만 설치돼 있었음
해결: %pip install fastapi requests → 서버 기동 성공 → GET /health 200 응답
교훈: 조용히 죽는 서버는 import부터 의심한다.
후속 조치: pip freeze > requirements.txt 갱신 (환경에 설치했으면 장부에 적는다 — Day 1 onnxscript 교훈의 재실전)





In [3]:
%pip install uvicorn
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('app', exist_ok=True)

_SERVERS = {} 

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0
    
def stop_server(port=8000):
    """실행중인 서버를 멈춤"""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='Warning'):
    """백그라운드 스레드에서 uvicorn 서버를 띄움"""
    stop_server(port)
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일 재저장 시 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def run():
        if sys.platform == 'win32':
            loop = asyncio.ProactorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for _ in range(40):
        if _port_open(host, port):
            print(f"서버 실행된: http://{host}:{port}")
            return server
        time.sleep(0.25)
    print("서버 실행 실패 위 로그 확인")
    return server

print("서버 도우미 준비 완료")


Note: you may need to restart the kernel to use updated packages.
서버 도우미 준비 완료



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
serve_in_thread("app.main_basic:app", port=8000)

서버 실행된: http://127.0.0.1:8000


In [46]:
%pip install fastapi requests

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp313-cp313-win_amd64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ----- ---------------------------------- 1/8 [pydantic-core]
   ---------- ----------------------------- 2/8 [anyio]
   ------------------------- -------------- 5/8 [starlette]
   ------------------------- ---------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import importlib
import app.main_basic
importlib.reload(app.main_basic)

In [8]:
import requests

res = requests.get("http://127.0.0.1:8000/health")
print(res.status_code)
print(res.json())

200
{'status': 'healthy'}


In [4]:
%%writefile app/main_params.py
from fastapi import FastAPI

app = FastAPI(title="Params Practice")

@app.get("/models/{model_name}")
def get_model(model_name: str):
    return {"model": model_name}

@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    return {"prediction_id": prediction_id}

@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    return {"status": status, "limit": limit}

Overwriting app/main_params.py


### 중간정리.
생각보다 작업이 자주 막혀서 한번 쉴겸 정리를 하고 가기로 했음
오늘 작업은 FastAPI 서버를 띄우고 첫 응답(200)을 받는 것까지였다. 모델 연결은 아직 안 했음
의존성 깔고 서버 켜면 끝일 줄 알았는데, "켠다"는 한 동작에 검문이 세 개 있었다:
1. 터미널이 어디 서 있는가 (cwd) → No module named 'app'
2. 어느 파이썬으로 실행하는가 (venv 활성화) → No module named 'fastapi'
3. 그 문이 비어 있는가 (포트 점유) → WinError 10013

In [89]:
print(requests.get(f"{BASE}/models").json())                          # 1. 아무것도 안 줌
print(requests.get(f"{BASE}/models?limit=5").json())                  # 2. limit만
print(requests.get(f"{BASE}/models?status=running&limit=3").json())  # 3. 둘 다
print(requests.get(f"{BASE}/models?limit=abc").status_code)          # 4. 또 부수기

{'detail': 'Not Found'}
{'detail': 'Not Found'}
{'detail': 'Not Found'}
404


In [ ]:
print(open("app/main_params.py").read())

from fastapi import FastAPI

app = FastAPI(title="Params Practice")

@app.get("/models/{model_name}")
def get_model(model_name: str):
    return {"model": model_name}

@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    return {"prediction_id": prediction_id}

@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    return {"status": status, "limit": limit}



: 

In [1]:
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/models").json())
print(requests.get(f"{BASE}/models?limit=5").json())
print(requests.get(f"{BASE}/models?status=running&limit=3").json())
print(requests.get(f"{BASE}/models?limit=abc").status_code)

{'status': None, 'limit': 10}
{'status': None, 'limit': 5}
{'status': 'running', 'limit': 3}
422


# ⏯ Day 2 재개 지점 (Quick Start)

2일차 진행중에 이해도 잘 안되고 학습이 잘 안되는것 같아서 우선 여기까지만 하고 체크포인트를 만들어서 퀵스타터를 만들어둠 


## 진행 상황
- ✅ §1 최소 서버 (main_basic.py, health 200)
- ✅ §2 Path + Query (main_params.py) — 422 자동 검증, None↔null 확인
- ⬜ §2 Body ← **여기서부터 재개**
- ⬜ §3 Swagger (/docs)
- ⬜ §4 Pydantic 검증
- ⬜ §5 모델 연결 (predict 엔드포인트 완성)

## 재가동 절차 — 터미널에서
1. `cd C:\Users\akals\Downloads\model-serving-course`
2. `.venv\Scripts\activate`   ← 프롬프트에 (.venv) 확인
3. `uvicorn app.main_params:app --host 127.0.0.1 --port 8000`
4. `Uvicorn running...`까지 확인 (ERROR 줄 없는지 끝까지 읽기)

## 서버 안 뜰 때 점검 순서
① 위치 맞나 (`dir`에 app 보여야) → ② venv 켰나 → ③ 포트 비었나
(`netstat -ano | findstr :8000` → 있으면 `taskkill /PID 번호 /F`)

## 주의사항 리마인드
- `%%writefile`은 셀 1행 1열, 셀당 하나
- 파일 고치면 터미널 서버 Ctrl+C 후 재기동 (--reload 안 쓰는 중)
- 커널 재시작해도 터미널 서버는 살아있음 (별도 생명)

In [45]:
# ===== Day 2 Quick Start — 이 셀 하나로 재가동 =====
import os, subprocess, time, requests

# 1. 작업 위치 고정
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("📁", os.getcwd())

BASE = "http://127.0.0.1:8000"

# 2. 서버 살아있나 먼저 확인 (이미 떠 있으면 재기동 불필요)
def server_alive():
    try:
        return requests.get(f"{BASE}/models?limit=1", timeout=2).status_code == 200
    except requests.exceptions.RequestException:
        return False

if server_alive():
    print("✅ 서버 이미 가동 중 — 바로 작업 시작")
else:
    # 3. 새 터미널 창을 열어 서버 기동 (venv 파이썬으로 직접 실행 → activate 불필요)
    print("🚀 서버 기동 중...")
    subprocess.Popen(
        'start "uvicorn-server" cmd /k ".venv\\Scripts\\python.exe -m uvicorn app.main_params:app --host 127.0.0.1 --port 8000"',
        shell=True,
    )
    # 4. 뜰 때까지 대기 후 확인
    for _ in range(20):
        time.sleep(0.5)
        if server_alive():
            print("✅ 서버 가동 + 응답 확인 완료")
            break
    else:
        print("❌ 안 떴음 — 새로 열린 터미널 창의 에러 메시지 확인 (포트 점유면: netstat -ano | findstr :8000)")

📁 c:\Users\akals\Downloads\model-serving-course
🚀 서버 기동 중...
✅ 서버 가동 + 응답 확인 완료


### 바디제작
class PredictRequest(BaseModel) > 본문은 이 양ㅇ식이어야 한다는 선언 text 필수 return_probabilities는 선택 기본값은 false고 쿼리일 때 기본값 = 생략가능이 됨 
@app.post > 데이터를 제춣하는 동작 본문을 post에 싣는것 
request: PredictRequest > 함수 인자에 클래스를 타입으로 

In [2]:
%%writefile -a app/main_params.py

from pydantic import BaseModel

class PredictRequest(BaseModel):
    text: str
    return_probabilities: bool = False

@app.post("/predict")
def predict(request: PredictRequest):
    return {"text": request.text, 
            "return_probabilities": request.return_probabilities,
            }


Appending to app/main_params.py


In [7]:
# 정상 — 본문에 JSON을 실어 보낸다
res = requests.post(f"{BASE}/predict", json={"text": "이 영화 정말 재밌다"})
print(res.status_code, res.json())

# 옵션까지
res = requests.post(f"{BASE}/predict", json={"text": "안녕", "return_probabilities": True})
print(res.status_code, res.json())

# 부수기 — 필수 필드 text 누락
res = requests.post(f"{BASE}/predict", json={"return_probabilities": True})
print(res.status_code, res.json())

200 {'text': '이 영화 정말 재밌다', 'return_probabilities': False}
200 {'text': '안녕', 'return_probabilities': True}
422 {'detail': [{'type': 'missing', 'loc': ['body', 'text'], 'msg': 'Field required', 'input': {'return_probabilities': True}}]}


### 

text: str 의 부분에서 {"text": ""}  처럼 빈 문자열을 보내도 통과는 된다 이건 ""도 문자열이기 때문이다. 
빈 문자열은 빈 텐서를 만들고 모델이 무의미한 출력을 뱉거나 어딘가에서 충돌이 일어나기도 한다. 
그래서 타입 검사에 추가로 조건을 단다. 

Field(..., min_length=1, max_length=5000)

... > 필수라는 기호값, 
min_length= 1 > 빈 문자열을 차단 
max_length=5000 너무 긴 문자열을 차단 


In [24]:
%%writefile app/main_params.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI(title="Params Practice")

@app.get("/models/{model_name}")
def get_model(model_name: str):
    return {"model": model_name}

@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    return {"prediction_id": prediction_id}

@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    return {"status": status, "limit": limit}

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    lang: Literal["ko", "en"] = "ko"
    return_probabilities: bool = False

@app.post("/predict")
def predict(request: PredictRequest):
    return {
        "text": request.text,
        "lang": request.lang,
        "return_probabilities": request.return_probabilities,
    }    

Overwriting app/main_params.py


In [25]:
print(requests.post(f"{BASE}/predict", json={"text": ""}).status_code)          # 아까는 200이었던 놈
print(requests.post(f"{BASE}/predict", json={"text": "정상 입력"}).status_code)
res = requests.post(f"{BASE}/predict", json={"text": ""})
print(res.json()["detail"][0]["type"], res.json()["detail"][0]["msg"])          # 거절 사유 확인

422
200
string_too_short String should have at least 1 character


In [26]:
# 1. 허용된 값
res = requests.post(f"{BASE}/predict", json={"text": "hello", "lang": "en"})
print(res.status_code, res.json())

# 2. 명단에 없는 값
res = requests.post(f"{BASE}/predict", json={"text": "bonjour", "lang": "fr"})
print(res.status_code, res.json()["detail"][0]["msg"])

200 {'text': 'hello', 'lang': 'en', 'return_probabilities': False}
422 Input should be 'ko' or 'en'


### 모델 연결하기. 
최종 산출물을 만들기 위한 준비를 한다. 
이전까지 만들었던 재료들을 쓰게 된다 우선은 app에서 model_utils.py, schemas.py, main.py  이렇게 3개로 나눠 쪼개서 분리하게 된다. 추론, 계약, 메인 3종류로 나눔 


In [34]:
%%writefile app/model_utils.py
"""모델 로드·추론 담당 — Day 1의 산출물을 서버 부품으로"""
import torch
import torch.nn as nn


class SimpleClassifier(nn.Module):
    """Day 1에서 학습한 모델과 동일한 구조 (fc 기반)"""
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(x)


def load_model(model_path: str = "models/model.pth") -> nn.Module:
    """state_dict를 읽어 추론 가능한 모델로 부활시킨다 (서버 시작 시 1회 호출)"""
    model = SimpleClassifier()
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    return model


def predict(model: nn.Module, input_tensor: torch.Tensor) -> dict:
    """전처리된 텐서를 받아 예측 결과를 dict로 반환"""
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, dim=1)

    return {
        "label": predicted.item(),
        "confidence": round(confidence.item(), 4),
        "probabilities": probabilities[0].tolist(),
    }

Overwriting app/model_utils.py


In [33]:
from app.model_utils import load_model, predict
import torch

m = load_model()
test = torch.randn(1, 1, 28, 28)
print(predict(m, test))

{'label': 5, 'confidence': 0.5401, 'probabilities': [0.0005099920090287924, 6.441691220970824e-05, 0.33068135380744934, 0.0594651959836483, 3.068905061809346e-05, 0.540098249912262, 1.6798947399365716e-05, 3.064544216613285e-05, 0.06910169124603271, 9.634214848119882e-07]}


### schemas.py 작성 

API입출력 계약
list[float] +min, max length의 값을 같은 값으로 가진다. 
출력 역시 ge, le, 0.0 1.0으로 숫자의 확률 범위에서 벗어나면 버그라고 스스로 검증하게 한다. 
Optional[list[float]] = None 이건 있을수도, 없을수도 없는 필드로 리턴값에 의해서 채워질수도 있고 빌수도 있는 값이 된다. 



In [37]:
%%writefile app/schemas.py
"""API 입출력 계약"""
from pydantic import BaseModel, Field
from typing import Optional


class PredictRequest(BaseModel):
    pixel_values: list[float] = Field(
        ...,
        min_length=784,
        max_length=784,
        description="28x28 이미지의 픽셀 값 784개",
    )
    return_probabilities: bool = Field(
        default=False,
        description="True면 10개 클래스 전체 확률 포함",
    )


class PredictResponse(BaseModel):
    label: int = Field(description="예측 숫자 (0~9)")
    confidence: float = Field(ge=0.0, le=1.0, description="확신도")
    probabilities: Optional[list[float]] = Field(
        default=None,
        description="클래스별 확률 (요청 시에만)",
    )


class HealthResponse(BaseModel):
    status: str
    model_loaded: bool

Writing app/schemas.py


In [38]:
import os
print(os.listdir('app'))                     # schemas.py 등장했는지
from app.schemas import PredictRequest, PredictResponse, HealthResponse
print("계약서 3종 준비 완료")

['main_basic.py', 'main_params.py', 'model_utils.py', 'schemas.py', 'test_repro.py', '__pycache__']
계약서 3종 준비 완료


In [39]:
%%writefile app/main.py
"""Day 2 최종: 모델 추론 API 서버"""
from fastapi import FastAPI, HTTPException
import torch

from app.model_utils import load_model, predict
from app.schemas import PredictRequest, PredictResponse, HealthResponse

app = FastAPI(
    title="MNIST Prediction API",
    description="Day 2 실습: 손글씨 숫자 분류 추론 API",
    version="1.0.0",
)

# ===== 모델 로드: 서버 시작 시 단 1회 =====
try:
    model = load_model("models/model.pth")
    model_loaded = True
    print("모델 로드 완료")
except Exception as e:
    model = None
    model_loaded = False
    print(f"모델 로드 실패: {e}")


@app.get("/health", response_model=HealthResponse)
def health_check():
    return HealthResponse(status="healthy", model_loaded=model_loaded)


@app.post("/predict", response_model=PredictResponse)
def predict_digit(request: PredictRequest):
    # 1. 모델 준비됐나
    if not model_loaded:
        raise HTTPException(status_code=503, detail="모델이 로드되지 않았습니다")

    # 2. 픽셀 784개 → 텐서 (1, 1, 28, 28)
    input_tensor = torch.tensor(request.pixel_values, dtype=torch.float32)
    input_tensor = input_tensor.reshape(1, 1, 28, 28)

    # 3. 추론
    try:
        result = predict(model, input_tensor)
    except Exception:
        raise HTTPException(status_code=500, detail="추론 중 에러가 발생했습니다")

    # 4. 응답 조립 (주문서 확인)
    response = PredictResponse(
        label=result["label"],
        confidence=result["confidence"],
    )
    if request.return_probabilities:
        response.probabilities = [round(p, 4) for p in result["probabilities"]]

    return response

Writing app/main.py


In [47]:
import os
print('main.py' in os.listdir('app'))   # True여야 함

True


In [48]:
from torchvision import datasets, transforms

# 어제 받아둔 MNIST 테스트셋에서 한 장
test_data = datasets.MNIST(root="data", train=False, download=True,
                           transform=transforms.ToTensor())
image, true_label = test_data[0]          # 첫 번째 이미지와 정답
print("정답:", true_label, "| 텐서 모양:", image.shape)

# 텐서 → 픽셀 784개 리스트 (JSON에 실을 수 있는 형태로)
pixel_values = image.reshape(-1).tolist()
print("픽셀 개수:", len(pixel_values))

정답: 7 | 텐서 모양: torch.Size([1, 28, 28])
픽셀 개수: 784


In [49]:
res = requests.post(f"{BASE}/predict", json={"pixel_values": pixel_values})
print(res.status_code, res.json())
print("정답은:", true_label)

200 {'label': 7, 'confidence': 0.9932, 'probabilities': None}
정답은: 7


In [50]:
# 3단계: 곱빼기 주문
res = requests.post(f"{BASE}/predict",
                    json={"pixel_values": pixel_values, "return_probabilities": True})
r = res.json()
print("label:", r["label"], "| probabilities:", r["probabilities"])

# 4단계: 연속 5장 주행
for i in range(5):
    img, label = test_data[i]
    res = requests.post(f"{BASE}/predict",
                        json={"pixel_values": img.reshape(-1).tolist()})
    pred = res.json()["label"]
    print(f"이미지 {i}: 정답 {label} → 예측 {pred} {'✅' if pred == label else '❌'}")

label: 7 | probabilities: [0.0001, 0.0, 0.001, 0.005, 0.0, 0.0001, 0.0, 0.9932, 0.0, 0.0006]
이미지 0: 정답 7 → 예측 7 ✅
이미지 1: 정답 2 → 예측 2 ✅
이미지 2: 정답 1 → 예측 1 ✅
이미지 3: 정답 0 → 예측 0 ✅
이미지 4: 정답 4 → 예측 4 ✅


In [51]:
# 파괴 1: 픽셀 783개 (1개 부족)
res = requests.post(f"{BASE}/predict", json={"pixel_values": pixel_values[:783]})
print("1) 783개:", res.status_code, res.json()["detail"][0]["type"])

# 파괴 2: 숫자 자리에 문자열
bad = pixel_values.copy(); bad[0] = "hello"
res = requests.post(f"{BASE}/predict", json={"pixel_values": bad})
print("2) 문자 픽셀:", res.status_code, res.json()["detail"][0]["loc"])

# 파괴 3: 필수 필드 누락
res = requests.post(f"{BASE}/predict", json={"return_probabilities": True})
print("3) 필드 누락:", res.status_code, res.json()["detail"][0]["type"])

# 파괴 4: 빈 JSON
res = requests.post(f"{BASE}/predict", json={})
print("4) 빈 JSON:", res.status_code)

# 생존 확인: 폭격 후에도 정상 영업하는가
res = requests.post(f"{BASE}/predict", json={"pixel_values": pixel_values})
print("생존 확인:", res.status_code, res.json()["label"])

1) 783개: 422 too_short
2) 문자 픽셀: 422 ['body', 'pixel_values', 0]
3) 필드 누락: 422 missing
4) 빈 JSON: 422
생존 확인: 200 7


### 2번쨰 마무리 

model = load_model("models/model.pth") 
서버가 켜질 때 한번 실행해야한다. try나 except로 감싸서 혹시나 모델로드가 되지 않았을 때는 오류메시지로 향하게 한다. 
/predict로 주문을 받으면 모델 로드에서 확인, 리스트에서 텐서로 번역, 그리고 이 값은 모델에게 전달, 
predict(model, tensor)에서는 
모델 추론 중에 혹시나 모를 에러가 생겨도 다시 try,except로 감싸 오류 메시지를 출력하게 한다. 
마지막으로 이 과정이 순조롭게 진행된다면 PredictResponse에 내용을 담아 출력하고 옵션이 있다면 옵션의 내용도 채워준다. 

